# CS124 Programming Assignment 1: RegEx and BPE (`Winter 2026`)

This assignment consists of two sections: **Regular Expressions** and **BPE Tokenization**.

In the first section, you will use regular expressions to extract email addresses from web documents.
In the second section, you will implement a BPE tokenizer from scratch.

If you need a refresher on `Python` or `Jupyter Notebooks` (or you are new to
either of them), we strongly encourage taking a look at `PA0` first.
Referring back to it as you work might help if you run into any issues with
`Python` syntax or idioms.
We also provide you with a `Regular Expressions` tutorial (`regular_expressions_tutorial.ipynb`) along with this assignment, which can be helpful for practicing regular expressions.

**You are encouraged to work with a partner!** We want the assignments in `CS 124` to bring you joy.
One way to ensure this is to work with a partner!
You are free to work with one other partner in our assignments.
If you choose to work with a partner, we ask that each partner work on each part of the assignment in jointly instead of splitting parts.
The partnership decision is independent for each assignment, so you can choose to work alone, work with the same partner or work with a different partner in the future assignments, which is a good way to meet your fellow classmates!

<a id="submitting"></a>
## Submitting

**Submit your empty assignment to Gradescope now to see the autograder output!**
You will submit your assignment via [`Gradescope`](www.gradescope.com), where we have an autograder set up.
You can submit your assignment any number of times before the deadline.
As a general rule of thumb, we recommend submitting early and often in any `Computer Science` class if you have the option, to prevent any last minute errors with autograders.
Submitting early also helps gauge how you are doing on the visible test cases of the autograder and gives you a chance to fix your submission accordingly.
In fact, start with submitting your assignment now (even if you haven't coded anything), so that you are familiar with the submission process and know what kind of autograder feedback is available to you.
You can re-submit as you make progress.
Don't forget to update your submission with your final version once you are done!

**Partners.**
You are welcome (and encouraged) to work with one partner.
If you do work with a partner, only one of you needs to submit the assignment on `Gradescope` and tag the other as a group member.

**Environment.**
Before you submit, make sure your code works in the environment described in the [`Environment Check`](#environment_check) section, as this is the environment our autograder will be run on.
If you have completed the setup steps in `PA0` and run this notebook in the `cs124` environment you created according to the instructions, you are good!
Note that you must not use any other dependencies (such as other `Python` modules), as doing so may cause the autograder to fail!

**Saving Your Notebook**.
Make sure to save the recent changes in your notebook (Ctrl + C on Windows and Cmd + C on Mac) before you run the submission script.





**Files.**
Once you are done, you only need to submit the file listed below.
**DO NOT** alter the file name.
```
pa1.ipynb
```

**Custom Dependencies.**
Sometimes you may want to put parts of your code into `.py` files and call them from your notebook instead of having all your functions in the notebook, or utilize extra datasets.
If this is the case, please put your extra files in a folder
named `deps/` (this folder should be on the same level as `pa1.ipynb`)
and upload a `zip` file (any name is fine) containing this folder and
`pa1.ipynb` to submit on `Gradescope`.
Note that these should be at the top directory of the `.zip` file (e.g. they should not be in a directory in the `.zip` file, as this will lead our autograder to fail at finding them).
To prevent this, ensure that you are only zipping the items mentioned, and not the folder containing them.
`Gradescope` will then automatically `unzip` the folder so that your
submission contains the following.
```
deps/
pa1.ipynb
```

**Submission Script.**
For your convenience, we are providing the following submission script that lets you automatically create a `zip` file to submit.
Simply run it and submit `submission.zip` to `Gradescope`.
Note that the script assumes that you have the `zip` utility installed.
You would need to install it if you don't already have it.

In [5]:
%%bash

if [[ ! -f "./pa1.ipynb" ]]
then
    echo "WARNING: Did not find notebook in Jupyter working directory. This probably means you're running on Google Colab. You'll need to go to File->Download .ipynb to download your notebok and other files, then zip them locally. See the README for more information."
else
    echo "Found notebook file, creating submission zip..."
    zip -r submission.zip pa1.ipynb deps/
fi

Found notebook file, creating submission zip...
  adding: pa1.ipynb (deflated 73%)
  adding: deps/ (stored 0%)
  adding: deps/example_dep.txt (stored 0%)


**Autograder.**
Once you submit, double check the autograder output to ensure that your submission didn't cause any error.

<a id="environment_check"></a>
## Environment Check

This assignment assumes that you have correctly set up the `cs124` conda environment and installed the required `Python` modules.
The cell below checks that you are running the correct version of `Python` and activated the `cs124` conda environment.
If you get an error running this cell, it means that you are either using the wrong `Conda` environment
or Python version!
If the latter, please exit this notebook, kill the notebook server with `CTRL-C`, and
try running:

`$ conda activate cs124`

Then restarting your notebook server with

`$ jupyter notebook`

If this doesn't work, you should go back and follow the installation instructions in `PA0`.

In [ ]:
import os
try:
    assert os.environ['CONDA_DEFAULT_ENV'] == "cs124"
except (KeyError, AssertionError):
    pass  # Skip check in autograder environment

import sys
try:
    assert sys.version_info.major == 3 and sys.version_info.minor >= 10
except AssertionError:
    pass  # Skip version check in autograder environment

<a id="setup"></a>
## Setup

**Getting the Necessary Files.** The cell below downloads the necessary files we will use in this assignment, if you don't already have them.

In [ ]:
%%bash

# Check if the ./data folder exists.
# Download it if not found.
if [[ ! -d "./data" ]]
then
    echo "Missing extra files. Downloading..."
    git clone https://github.com/cs124/pa1-spamlord.git
    cp -r ./pa1-regexes/{data,deps,util.py} .
fi

**Importing Modules.** Run the next cell to import the necessary modules we will use in this assignment.

In [ ]:
""" Modules included in the Python Standard Library """

# We use features from io and os modules for opening files and writing to them
from io import open
import os

# re module contain methods for using regular expressions
import re

# typing module contains type objects. We will use these types to ensure that 
# the inputs and outputs passed to the functions you will be implementing are 
# of the correct type
from typing import List

In [ ]:
""" Our custom functions and classes """

# Helper functions we will use later
from util import process_dir, get_gold, score

**Note:** **DO NOT** import and use any other packages outside of the Python standard
library. Although we provide `NumPy`, `scikit-learn`, and other packages in
the `Conda` environment we set up for you, you will not be using them in this assignment, only
in later assignments. Importing them in your solution will cause it to fail the
autograder.

# Section A: Regular Expressions

This part of the assignment is your chance to become a __Dark Lord__ of spam email!
Yes, you too can build regular expressions (`RegExes`) to spread evil throughout the galaxy. 
Our goal in this part is to use `RegExes` to extract
email addresses from documents found on the web.
This may seem easy at first, as you can write very simple `RegExes` to catch similar cases such as `manning@cs.stanford.edu`.
On the other hand there are various different ways people write their emails in `HTML` documents, some to prevent scrapers from capturing them easily, which you will learn in more detail in the upcoming sections.
If you really were a malicious actor, you could then use these extracted addresses to bombard unsuspecting victims with spam!

Of course, we would never do anything nefarious like that in `CS 124`. 
Instead our goal will be to work with raw data and gain some experience with `RegExes`.

<a id="contents"></a>
## Contents

Listed below are the contents of the Regex portion. In the `Data Exploration` part, you will look into the dataset we will use in this section. In the `Example Approach` part, you will learn more about the specifics of our email address catching task, and implement see an example implementation. In the `Evaluation` part, you will learn how to evaluate `RegExes` on our dataset. `Cases to Consider` part provides you with tips on the tricky cases you may run into. `Your Approach` part is the place where you actually start coding. In the `Reflection` part, you will answer a few questions on the environmental and societal impacts of spamming. Please read through all of Section A: Regular Expressions before you start working through this section.

* [`Part 1. Data Exploration`](#data_exploration)
* [`Part 2. Example Approach`](#example_approach)
* [`Part 3. Evaluation`](#evaluation)
* [`Part 4. Cases to Consider`](#cases_to_consider)
* [`Part 5. Your Approach`](#your_approach)
* [`Part 6. Reflection`](#reflection)



<a id="roadmap"></a>
## Roadmap

As an overview, there are only `3` functions you need to implement in this section:
* In `Part 5. Your Approach`: **[`find_emails()`](#your_approach)**
* In `Part 6. Reflection`: **[`calculate_attention_tax()`](#academic_commons)** and **[`fairness_response()`](#fairness_response)**

You will write your `RegExes` in **`find_emails()`**, which makes up the meat of this section and will take the longest. A very short implementation is needed for the **`calculate_attention_tax()`** function. You will provide a short answer to an open-ended question in the **`fairness_response()`** function.

<a id="data_exploration"></a>
## Part 1. Data Exploration

Let's start by taking a look at what our data actually looks like.
This should always be one of the first things you do whenever you are solving a problem that requires working with data.

**Development Set.** In order to make your life easier on this and future homeworks, we will be
giving you some data to study and test your code on, which we call a
`development set` or a `dev set`.
Using a dev set to test and evaluate your methods is an extremely common approach in `Natural Language Processing` and `Machine Learning`.
More generally, coming up with a robust set of test cases to evaluate your work against is an extremely important part of writing good code.

Our dev set consists of a bunch of `HTML` documents (the personal
homepages of some `Stanford CS` professors) that we have scraped from the web and downloaded for you. 
If you are not familiar with the details of `HTML` or its syntax, it's fine. 
For the purposes of this assignment, all you need to know is that the inputs are text files (with some formatting) that contain the (possibly obfuscated) emails that we want to extract.
You can find all of these `HTML` documents in the `data/dev` directory.

**Exploration.** To visualize the documents in our dev set, you can take any of the files and open them in a browser of your choice!
For example, by right-clicking on `data/dev/dabo.html` and clicking `open with -> Firefox`.
You can also try double-clicking, which usually opens the page in your default browser.
Feel free to change the filename to some of the other files in `data/dev`
(i.e. `dabo` to `aiken`, `balaji`, etc.) to take a look at some of the other
faculty pages.
```
Mini Task: Open data/dev/dabo.html, and look for email addresses.
What kind of regular expressions you would need to catch these?
```
You should find that, as expected, the page you opened looks exactly like a
faculty webpage, possibly minus some images which we didn't download along
with the `HTML`, but that's fine, as we are only interested in the text.
As is common for faculty pages, these pages have contact information like email
addresses. Our goal is to write regular expressions
that we can use to automatically match and extract these from the webpages.

**Reading the HTML Documents.**
We have seen what the files look like as webpages.
However, we are interested in the text contents, as that is what we will be matching with our regular expressions.
Let's try reading in the `data/dev/dabo.html` as a single giant text string.

In [ ]:
# Open and read a file as a gigantic string
with open("data/dev/dabo.html", 'r', encoding='ISO-8859-1') as file:
    full_text = file.read()
    print(full_text)

Okay, it's a bit long and hard to parse, but it seems reasonable!
There's a bunch of somewhat cluttered `HTML` markup, but it's all in text form and if we search through it, it looks like all of the text from the page including the emails, is somewhere in there.
In the remainder of this section, you will come back to printing the strings for the `HTML` documents to understand the cases to improve your regular expressions by finding the cases that they are missing.

<a id="example_approach"></a>
## Part 2. Example Approach

Now that we have learned how to load `HTML` documents from our dataset and inspect them, let's see if we can extract some email addresses using a regular expression pattern.
In this case we will use a super-simple pattern that just looks for 1 or more alphanumeric characters or periods followed by an `@` followed by 1 or more
alphanumeric characters or periods, followed by `.edu`. This is just the usual
format of an email address.
```
leland1@stanford.edu
```
We can achieve our goal with the following regular expression.
```
([\w\.]+)@([\w\.]+\.edu)
```
Let's break down what this pattern does!
* `\w`: A word character, same as the regular expression `[A-Za-z0-9_]`. Note that it matches `_` too!
* `[\w.]`: Matches any word character or `.`. Note that we don't need to use an escape character before `.`, since any character other than `^`, `-`, `\` or `]` is interpreted as a literal in a character class (which is denoted by `[]`).
* `[\w\.]+`: Matches at least 1 word character or period. This is the pattern we wanted to match for the name part of the regular expression, so we are done!
* `[\w\.]+\.edu`: Matches at least 1 word character or period followed by `.edu`. Note that we have to use an escape character before the period this time around.
* `(...)`: Capture groups for saving the matches.
That is, the regular expression engine will not only match the expression inside `()` to a part of an expression, but also record the matched part in a capture group, which we can retrieve later.
* `([\w\.]+)` and `([\w\.]+.edu)`: The capture groups are used to capture the part of the email before and after the `@`, respectively.

Let's test our pattern on a short string.
Notice how we use the formatter character `%` in combination with tuples to build strings in our desired format.


In [ ]:
"""
The function re.findall takes a regex pattern and a text string and returns all 
matches in the string as a list. Each match in the list is a tuple of the 
capture groups in the expression. So in this case, each element in matches will 
be a tuple of form:

    (stuff before '@', stuff after '@')

"""
# Create the example string and patter
example = 'The email address is leland1@stanford.edu.'
simple_pattern = '([\w\.]+)@([\w\.]+\.edu)'

# Find the matches, which are returned as a list containing two-tuples
matches = re.findall(simple_pattern, example)

# Iterate over the matches 
for m in matches:
    # Print matches
    print("The first capture group is: %s" % m[0])
    print("The second capture group is: %s" % m[1])
    print("The first and second capture groups are: %s and %s" % m)

    # Put the email back together
    email = '%s@%s' % m
    print(email)

Observe how we put the email back together using capture groups.
We can now wrap the same code above in a function, that takes in a sring and returns the list of emails found in the string.

In [ ]:
# Define our function
def example_find_emails(full_text: str) -> List[str]:
    """
    This is an example function that takes a string and finds the emails in
    it. Returns the found emails in a list of strings. The returned emails
    must follow the canonical format:

              'someone@something'

    We use -> to show the return type of the function. Typing isn't explicitly 
    enforced in Python, so we didn't have to specify the return type of our 
    function, but we are specifying them in this assignment to help you tackle
    errors in an easier way.

    full_text (str): Full text of the html file read.
    """
    # The simple pattern
    simple_pattern = '([\w\.]+)@([\w\.]+\.edu)'
    matches = re.findall(simple_pattern, full_text)

    # Iterate over the matches
    res = []
    for m in matches:
        email = '%s@%s' % m
        res.append(email)
    return res

Let's see if our function works as expected.

In [ ]:
# Call our function
example_line = 'The email address is leland1@stanford.edu.'
example_find_emails(example_line)

Great! We now have a simple function that we can call on a string to extract
email in simple forms.

<a id="evaluation"></a>
## Part 3. Evaluation

Evaluation is a crucial step of any kind of `NLP` or `ML` project.
For us to be able to evaluate how our functions are doing, we need some sort of grounding.
In addition to the `HTML` documents, the `data` directory also contains
another file, `data/devGOLD`.
You can think of this file as the answer key corresponding to the documents in `data/dev`.
It contains all the correctly extracted emails from all the documents in `data/dev`, in a particular format so your scripts as well as our grading scripts can read them easily.

### Part 3.1. Format Matches


Each line in the `data/devGOLD` file represents one extracted email address
in the form of a 3-tuple. 
Each tuple is represented as 3 strings separated by vertical bars ("|"). 
You can open the `data/devGOLD` file to see for yourself.

```
jurafsky|e|jurafsky@stanford.edu
```

* The **first** string is the name of the file that the match came from where the `.html` extension removed.
* The **second** string is an `e` indicating the match was an email address.
* The **third** string is the actual extracted email address itself,
in the following canonical form.

```
  user@example.com
```

To sum up, the answers in the ```data/devGOLD``` file and the outputs
generated by your implementation should take the form of `Python` tuples
that look like the following.

```
  (filename, match type, match value)
```

The functions we have coded so far can take in a string and return the list of extracted email addresses.
To be able to evaluate our functions, we need another function that will call our functions on each line of a file, and output the results in the specified format above. The function shared in the next cell, **`example_process_file()`** does exactly this.

__Note:__ You don't have to worry about case sensitivity in your values
(email addresses), since they will be normalized to lower case
before being compared with the answers.


In [ ]:
def example_process_file(filename: str, data_directory: str):
    """
    Function we wrote to call the functions listed below on each line of a file 
    with the given filename. It returns a list of 3-tuples representinting the 
    found matches in the specified evaluation format.
    
    * example_find_emails()

    """
    # The format of our evaluation matches requires stripping the ".html" 
    # extension from our filenames.
    filename_no_ext, ext = filename.split('.')
    absolute_file_path = os.path.join(data_directory, filename)
    res = []
    with open(absolute_file_path, 'r', encoding='ISO-8859-1') as file:
        # Read the full text
        full_text = file.read()
        # Call example_find_emails
        emails = [(filename_no_ext, 'e', e) for e in example_find_emails(full_text)]
        # Add the newly extracted emails to our list
        res += emails

    return res

Let's check which emails our functions will find in a given file.

In [ ]:
result = example_process_file('dabo.html', 'data/dev')
print(result)

Success! It looks like we got our first match! The output of our function in
this case is a list of matches, where each match is a tuple in the following format. The reason we want our output in this format is so that it works
with our automated scoring later.
```
(file, e indicating email, extracted email)
```

In this case we can see that we have just a single match from the file `data/dev/dabo` which is an email and is the address `dabo@cs.stanford.edu`.
Note that we only searched in this one file!

So far we have seen how we can process a single file.
However, our dev set consists of many such files.
We need a function to loop over all of them, process them, and return all the extracted addresses. 
This function is provided for you in `util.py`, and it is named **`process_dir()`**!
You shouldn't modify this function, but we encourage you to take a look at how it is implemented.

In [ ]:
all_results = process_dir('data/dev', example_process_file)

print(all_results)

Looks like we got quite a few more matches, even with our very simple pattern. You may have also noticed that our results have quite a few duplicates. If you
examine the corresponding files, you can see that this is happening because
the same email address appears more than once in the file. Don't worry about this for now, we wil be careful to strip out duplicates later when we are doing our scoring.

### Part 3.2. Compare to Gold

The final step of the evaluation process is straightforward: all that needs to be done is to load the correct answers for the dev set from the provided file (`data/devGOLD`) and compare them to the matches that were generated by our function. 
We provide this helper function in `util.py`: it is called **`get_gold()`**.
Again, you shouldn't modify it, but you can take a peek at it if you are curious what it's doing. Let's use it to read the gold (correct) matches from the provided file.

In [ ]:
all_gold_matches = get_gold('data/devGOLD')
print(all_gold_matches)

As expected, these exactly match the output format that we showed earlier for the file processing function.
This will make it easy to compare our answers to the gold answers, for which we provide a helper function in `util.py`, called **`score()`**.
You are welcome to take a look if you are curious. 
It takes in a list of your predicted matches, the output of the function you will write, and a list of correct/gold matches, read from the `data/devGOLD` file.
It compares the two and calculates how they overlap, printing out a bunch of information in the following form:

```
  True Positives (4):
  set([('balaji', 'e', 'balaji@stanford.edu'),
       ('nass', 'e', 'nass@stanford.edu'),
       ('shoham', 'e', 'shoham@stanford.edu'),
       ('thm', 'e', 'pkrokel@stanford.edu')])
  False Positives (1):
  set([('psyoung', 'e', 'young@stanford.edu')])
  False Negatives (113):
  set([('ashishg', 'e', 'ashishg@stanford.edu'),
       ('ashishg', 'e', 'rozm@stanford.edu'),
  ...
```

You can interpret the results as follows:

* **`The true positive`** section displays emails which are in
both your list of matches and the gold matches list.
These are examples that your regular expressions correctly found.

* **`The false positive`** section displays matches which your regular expressions extracted but which are not in the gold matches list.
These are incorrect and show where your method may have been too
broad/aggressive.

* **`The false negative`** section displays emails which your code did not match, but which do exist in the html files.
These are the matches your code missed.

Your goal, then, is to reduce the number of false positives and false negatives
to 0.
At the bottom of the output you can see the total counts of `true positives`, `false positives`, and `false negatives`.

Let's try evaluating our existing super-basic method using the method described above.

In [ ]:
guess_list = process_dir('data/dev', example_process_file)
gold_list = get_gold('data/devGOLD')
score(guess_list, gold_list)

Looks reasonable! It appears that our basic method produced 27 matches,
while the gold set contains 110 matches.
There were: 
* 27 `true positives`, which are matches that we found that were in the gold set;
* 0 `false positives`, which are matches that we found that were NOT in the gold set;
* 11 `false negatives`, which are matches in the gold set that we did NOT find.

This seems like a pretty good start, but there are still 11 addresses that our
approach didn't manage to catch.
Figuring out how to extract these addresses without accidentally matching any non-address text is up to you!


### Part 3.3. Point Distribution

* The **first** part, worth 8 points, scores how well your implementation does on the
development set.
For these examples you're given the correct answers, so you should aim to get 100% of them correct!

* The **second** part of your grade, worth 4 points, will be based on how well your
regular expressions find emails in a different set of
examples, the `test set`. 
This test set is hidden and only the teaching staff knows what is in it!
Because you don't know exactly what trickery goes on in this test set, you should be creative in thinking of different ways of writing (and hiding) emails.

* The **third** part is a brief section worth 1 point (0.5 for CO2 calculation + 0.5 for free response) designed to get you thinking about ethical issues surrounding spam emails.

You are not expected to perform perfectly on the test set as you don't know
what is in it, or have the correct answers (just like in real life).
As long as you manage to achieve some reasonable performance (compared to a benchmark that we provide), you will get full points! 
The benchmark is set at **42** test errors or fewer.
Normally, we would hide your test set performance so you can't tune your methods
to maximize test set performance (this is good experimental procedure).
However, in the interests of transparency and making your life easier, we will show you your test score and number of test errors on `Gradescope` so you can get an idea of how close you are to the benchmark and full points.

You are free to submit as many times as you'd like on `Gradescope` until you hit
the benchmark (or beat it!).

Here are the equations we use to calculate the scores for the two parts, where
`e` is the total number of errors (`false negatives` and `false positives`) for
each part:

__Dev:__

```
  if e < 5 then score(e) = 8 - e
  else if e >= 5 then score(e) = 3
```

__Test:__

```
  if e <= 42      then score(e) = 4
  else if 42 < e  then score(e) = 4 - (e - 42) * 0.1
```

__Note:__ This sort of two-stage evaluation (a known development set and a
hidden test set) is a very commonly used approach in machine learning!
Evaluating on a development set where we have the "right" answers lets us
measure our performance precisely and improve our approach, while a test set
that is hidden from us until later allows us to see how we perform
"out in the wild", on examples that we might not have been able to tailor
our methods to.

<a id="cases_to_consider"></a>
## Part 4. Cases to Consider

As you implement your regular expressions and analyze the `HTML` files that your approach isn't getting quite right, you will develop an understanding of which cases to consider.
Your development workflow will be as follows:
* You will start with a simple regular expression.
* You will evaluate your simple approach against the gold matches.
* You will find the files for which your approach is failing and try to identify how you can improve your regular expression to do better.
* You will go back to evaluation step and repeat until you are satisfied.

This is how a real life `NLP` or `ML` practioner would approach an unknown task!
For our assignment, to make things a little more concrete, we are providing you with some examples to illustrate exactly what your implementation should be able to do if it's working correctly.
The list we provide here is not comprehensive, so you may find out about cases that we haven't covered here.

### Part 4.1. Extracting Email Addresses

We are interested in processing text
containing (possibly obfuscated) email addresses and returning the corresponding
email addresses in a standard form.

```
# Ordinary email addresses
manning@cs.stanford.edu => manning@cs.stanford.edu

# Hidden email addresses
manning(at)cs.stanford.edu => manning@cs.stanford.edu
manning at csli dot stanford dot edu => manning@csli.stanford.edu
```
Below are some notes/questions to guide you. Make sure to account for different cases (lowercase, uppercase, mixed) for each of the following points!
* Notice the different ways people write the `@` sign. 
  Can you identify a few?
* What about the alternative ways of writing `.` in emails?
  Make sure to account for different cases (lowercase, uppercase, mixed)!
* What are some popular top level domain names?
  To get full credit on the section, it is sufficient to consider `com`, `gov`, `org`, `edu`.
  Remember to account for cases!
* Are there other ways people write their emails in plain english?
  What are some of the common ones?

### Part 4.2. Cases to not Worry About

Although you should aim to make your regexes as powerful and general-purpose as
you possibly can, there are some cases that are difficult or impossible to
handle with regexes and which we don't expect you to be able to deal with.

These include:

* Anything involving images or other non-text ways of displaying emails.
* Examples that require parsing names into parts, like:.

```
"first name"@cs.stanford.edu
```

* Particularly clever/difficult examples that don't contain much or any
part of the actual email address. For example,

```
To send me email, try the simplest address that makes sense.
```

<a id="your_approach"></a>
## Part 5. Your Approach

The example functions we shared so far only allows us to retrieve a subset of the present emails in our dataset.
In this section, you will implement your version of the example functions, and test your implementations 
Your task is to modify the function **`find_emails()`** given below.
We provide you with a placeholder code, but you will modify it.
Here we share some notes/tips that may be helpful in your implementations:
* You can use separate regular expressions for separate cases, and combine your results into a list before returning.
This will make writing regular expressions easier.
* You may get long regular expressions as you try to cover each email case.
Don't get discouraged and make use of `|`.
* Although they are mostly the same, different regular expression engines differ in subtle ways, especially true for the way escape characters etc. are interpreted.
If you are using an external website to test your `RegExes`, be aware that your `RegExes` may not work out of the box when you move them over to `Python` due to this distinction.

In [ ]:
# TODO: Implement your approach here!
def find_emails(full_text: str) -> List[str]:
    """
    Takes in a line from an html document as a string and finds the emails in
    it. Returns the found emails in a list of strings. The returned email
    must follow the canonical format:

              'someone@something'

    NOTE: DO NOT CHANGE THIS INTERFACE, as it will be called directly by
    the submit script.

    full_text (str): Full text of the html file read.
    """
    # CODE START
    return []
    # CODE END

In [ ]:
# DO NOT CHANGE
def process_file(filename: str, data_directory: str):
    """
    Function we wrote to call the functions listed below on each line of a file 
    with the given filename. It returns a list of 3-tuples representinting the 
    found matches in the specified evaluation format.
    
    * find_emails()

    """
    # DO NOT CHANGE
    filename_no_ext, ext = filename.split('.')
    absolute_file_path = os.path.join(data_directory, filename)
    res = []
    with open(absolute_file_path, 'r', encoding='ISO-8859-1') as file:
        # Read the full text
        full_text = file.read()
        
        # Call find_emails
        emails = [(filename_no_ext, 'e', e) for e in find_emails(full_text)]
        
        # Add the newly extracted emails to our list
        res += emails

    return res

Similar to the example functions, you can run your functions on all of the dev set and compare your found matches with the gold set matches.

In [ ]:
guess_list = process_dir('data/dev', process_file)
gold_list = get_gold('data/devGOLD')
score(guess_list, gold_list)

From the list above, select a file for which your approach is outputting an incorrect result, print the contents of this file using the next cell, and look into why your regular expression may not be capturing the missed emails.
As you make improvements to your functions, come back to this section and repeat the process and you are satisfied with the reuslts.

In [ ]:
selected_file = 'dabo.html'
result = process_file(selected_file, 'data/dev')
print(result)

<a id="reflection"></a>
## Part 6. Reflection

<a id='academic_commons'></a>
**The Academic Commons.** Stop! Before you use your new RegEx skills to scrape every faculty directory on campus, consider the "Academic Commons." In economics, a Common Pool Resource is a resource available to everyone (e.g., a clean lake or a professor’s inbox) that can be degraded by over-use. While it is tempting to use automation to "mass-blast" cold emails in hopes of securing a research assistantship, the cumulative effect of hundreds of students sending templated messages creates an "attention tax" that can lead to a tragedy of the commons. When an inbox is flooded with automated outreach, a professor’s limited time and attention are depleted, often causing them to ignore all cold emails entirely.

The Scenario:

Consider the following estimates for Stanford CS in 2025-2026:
- **The Class (N):** There are **300** students in CS 124.
- **The Faculty (M):** There are **100** active research faculty in the department.
- **The Tax (T):** It takes a professor **15 seconds** to read a subject line, realize a message is a templated "mass-email," and archive it.

If every student in CS 124 sends just one templated cold email to every CS professor at the start of the quarter, calculate the Total Faculty Time (in hours) consumed by the department just to process and delete these 30,000 emails.

Note: Use 1 hour = 3600 seconds.


In [ ]:
# TODO: Modify this function so that it returns your solution
def calculate_attention_tax():
    """
    Calculate the total hours of faculty time consumed if 
    300 students each email 100 professors, 
    and each email takes 15 seconds to delete.
    """
    total_hours = 0
    # CODE START
    
    # CODE END
    return total_hours


<a id='government_response'></a>
**Fairness in the Commons.** When the "Academic Commons" is flooded with automated outreach, it creates a **fairness challenge**. Professors may develop "email fatigue," leading them to ignore all cold emails, including those from students who spent hours researching a professor’s specific publications to write a thoughtful, personalized message.

Your Task:

Please provide a 3-6 sentence response addressing the following:
- How does the use of mass personalization by LLMs for cold emails create an unfair disadvantage for students who write highly personalized, non-automated messages?
- If professors stop responding to cold emails entirely due to high volume, which groups of students are most negatively impacted?



In [ ]:
# TODO: Place your response into the response string below
def fairness_response():
    response = ""
    return response

# Section B: BPE Tokenization

In this part of the assignment, you will implement a BPE tokenizer from scratch.
In particular, we will represent arbitrary (Unicode)
strings as a sequence of bytes and train our BPE tokenizer on this byte sequence. Later, we will use this
tokenizer to encode text (a string) into tokens (a sequence of integers) for language modeling.

Acknowledgement: This assignment is adapted from CS336.

<a id="contents_b"></a>
## Contents


Listed below are the contents of the BPE Tokenization portion. In the `The Unicode Standard` part, you will learn about Unicode code points and how characters are represented. In the `Unicode Encodings` part, you will learn about UTF-8 encoding and how to convert text to bytes. In the `Subword Tokenization` part, you will understand why subword tokenization is preferred over word-level or character-level approaches. In the `BPE Tokenizer Training` part, you will learn how to train a BPE tokenizer on a corpus. In the `Encoding Text` part, you will implement functions to encode text into token IDs. In the `Decoding Text` part, you will implement functions to decode token IDs back to text. Please read through all of Section B: BPE Tokenization before you start working through this section. 

* [`Part 1. The Unicode Standard`](#unicode_standard)
* [`Part 2. Unicode Encodings`](#unicode_encodings)
* [`Part 3. Subword Tokenization`](#subword_tokenization)
* [`Part 4. BPE Tokenizer Training`](#bpe_training)
* [`Part 5. Encoding Text`](#encoding_text)
* [`Part 6. Decoding Text`](#decoding_text)



<a id="contents_b"></a>
## Roadmap

As an overview, there are only `3` functions you need to implement in this section:
* In `Part 5. Encoding Text`: **[`encode_chunk_no_special_tokens()`](#encode_chunk_no_special_tokens)** and **[`encode()`](#encode)**
* In `Part 6. Decoding Text`: **[`decode()`](#decode)**.

Although you will only write code at the end of this section, you should read through the entire section (especially the code samples) to understand how BPE tokenization works, how bytes and merges are handled, and how the different variables fit together. A much shorter implementation is needed for **`decode()`**.


<a id="unicode_standard"></a>
## Part 1. The Unicode Standard

**Unicode** is a text encoding standard that maps characters to integer code points. As of Unicode 16.0 (released in Sep. 2024), the standard defines `154,998` characters across `168` scripts. 

For example, the character `“s”` has the code point `115` (typically notated as `U+0073`, where `U+` is a conventional prefix and `0073` is `115` in hexadecimal), and the character `“牛”` has the code point `29275`. 

In Python,
- The `ord()` function converts a single Unicode character into its integer representation.
- The `chr()` function converts an integer Unicode code point into a string with the corresponding character.

In [ ]:
# Example:
ord('牛')
chr(29275)

<a id="unicode_encodings"></a>
## Part 2. Unicode Encodings

While the Unicode standard defines a mapping from **characters to code points (integers)**, it’s impractical to
train tokenizers directly on Unicode codepoints, since the vocabulary would be **prohibitively large** (around
`150K` items) and **sparse** (since many characters are quite rare). 

Instead, we’ll use a `Unicode encoding`, which
converts a Unicode character into a **sequence of bytes**. The Unicode standard itself defines three encodings:
`UTF-8`, `UTF-16`, and `UTF-32`, with `UTF-8` being the dominant encoding for the Internet (more than 98%
of all webpages).

Unicode encodings in Python work as follows:
- To encode a Unicode string into `UTF-8`, we can use the `encode()` function in Python. 
- To access the underlying byte values for a Python bytes object, we can iterate over it (e.g., call `list()`). 
- Finally, we can use the `decode()` function to decode a `UTF-8` byte string into a Unicode string.

In [ ]:
# Example:
test_string = "hello! こんにちは!"
utf8_encoded = test_string.encode("utf-8")
print(utf8_encoded)
print(type(utf8_encoded))

In [ ]:
# Get the byte values for the encoded string (integers from 0 to 255).
print (list(utf8_encoded))
print (len(test_string))
print (len(utf8_encoded))
print (utf8_encoded.decode("utf-8"))

By converting our **Unicode** codepoints into a sequence of **bytes** (e.g., via the `UTF-8` encoding), we
are essentially taking a sequence of **codepoints** (integers in the range `0` to `154,997`) and transforming it
into a sequence of **byte values** (integers in the range `0` to `255`). The `256`-length byte vocabulary is much
more manageable to deal with. 

When using byte-level tokenization, we do not need to worry about **out-of-vocabulary** tokens, since we know that 
any input text can be expressed as a sequence of integers from `0` to `255`.

<a id="subword_tokenization"></a>
## Part 3. Subword Tokenization

While **byte-level tokenization** can alleviate the **out-of-vocabulary** issues faced by word-level tokenizers, tokenizing text into bytes results in extremely long input sequences that: 
- slows down model training (a sentence with 10 words might be 10 tokens in a word-level model but 50+ tokens in a byte-level model)
- requires more computation at each step
- creates longer dependencies in the data.

**Subword tokenization** is a midpoint between word-level and byte-level tokenizers. A subword tokenizer trades off a larger vocabulary size for better compression of the input byte sequence. For example, if the byte sequence `b"the"` often occurs in our training data, assigning it an entry in the vocabulary would reduce this `3-token sequence` to a `single token`.

**How do we select these subword units to add to our vocabulary?** Sennrich et al. [2016] propose to use `Byte Pair Encoding (BPE)` (Gage, 1994), a compression algorithm that *iteratively merges the most frequent pair of bytes with a single, new unused token*. If a word occurs in our input text enough times, it'll be represented as a single subword unit.

In this assignment, we'll implement a **byte-level BPE tokenizer**, where the vocabulary items are bytes or merged sequences of bytes, which gives us the best of both worlds: out-of-vocabulary handling and manageable input sequence lengths. The process of constructing the BPE tokenizer vocabulary is known as **"training"** the BPE tokenizer.

<a id="bpe_training"></a>
## Part 4. BPE Tokenizer Training

The BPE tokenizer training procedure consists of 3 main steps:
1. **Vocabulary initialization** (bytes + special tokens)
2. **Pre-tokenization**
3. **Perform BPE merges**

#### Vocabulary initialization 
For a byte-level BPE tokenizer, the initial vocabulary consists of all `256` possible byte values, each mapped to a unique token ID.

In [ ]:
# Initialize with all 256 possible byte values
vocab = {i: bytes([i]) for i in range(256)}

***Special tokens.*** Some strings (e.g., `<|endoftext|>`) encode metadata such as document boundaries and should always be treated as a single token. These “special tokens” are never split during encoding and are added to the vocabulary with fixed token IDs.

In [ ]:
# Add special tokens to the vocabulary
special_tokens = ["<|endoftext|>"]
for i, special_token in enumerate(special_tokens):
    vocab[256 + i] = special_token.encode("utf-8")

#### Pre-tokenization

Directly merging frequent byte pairs over the full corpus:
- is **computationally expensive**: it would take a full pass of the corpus each merge.
- can **produce redundant tokens** that differ only by punctuation (e.g., dog! vs. dog.) that may have different token IDs despite semantic similarity. 

To address this we **pre-tokenize** the corpus into coarse-grained tokenization, and then later count how often pairs of characters appear. For example, if the *pre-token* `text` appears `10` times, the pair (`t`, `e`) is incremented by `10` instead of rescanning the corpus. Since this is a byte-level BPE model, each *pre-token* is represented as a sequence of UTF-8 bytes.


The original BPE method (Sennrich et al., 2016) splits on `whitespace`. Instead, we use the GPT-2 `regex-based` pre-tokenizer (Radford et al., 2019) from github.com/openai/tiktoken/pull/234/files.

In [ ]:
import regex as re

## We will use this regex pattern to pre-tokenize the corpus
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
re.findall(PAT, "some text that i'll pre-tokenize")

In [ ]:
from collections import defaultdict, Counter

pattern = re.compile(
        r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""",
        re.UNICODE
    )

with open("data/corpus.en", "r") as f:
    corpus = f.read()

# Pre-tokenize the corpus
total_counter = Counter()
parts = corpus.split("<|endoftext|>") # Split on special tokens

# Counting how often each pre-token appears
for part in parts:
    if part:
        total_counter += Counter(re.findall(pattern, part))

print (total_counter)

# Convert string tokens to bytes and store counts
pre_token_counts = {}
for token, count in total_counter.items():
    pre_token_counts[token.encode('utf-8')] = count



#### Compute BPE merges
Once the corpus is pre-tokenized and each pre-token is represented as UTF-8 bytes, we "train" the BPE tokenizer by repeatedly merging the most frequent adjacent byte pairs. Each merge creates a new token and expands the vocabulary.

1. Count pairs. Count all adjacent byte pairs within each pre-token.
    - For efficiency, we do not consider pairs that cross pre-token boundaries.
2. Merge. Select the most frequent pair (“A”, “B”) and replace every occurrence with a new token “AB”, adding it to the `vocabulary`.
    - If multiple pairs tie, merge the lexicographically greatest pair (e.g., among (“A”, “B”), (“A”, “C”), (“B”, “ZZ”), and (“BA”, “A”), choose (“BA”, “A”)). For example, this line behlow selects the exicographically greatest pair:

          max([("A", "B"), ("A", "C"), ("B", "ZZ"), ("BA", "A")])
3. Repeat. The final vocabulary size is `initial vocabulary size (256 in our case) + the number of merges`. 

The next 4 code blocks below show the full implementation of training. *It has many components, and while you do not need to understand it in depth, it is helpful to understand the overall approach.* It is also important to note the `merges` variable: this stores the sequence of merge operations learned during training.

##### Setup for BPE Training
We first convert each pre-token in `pre_token_counts` into a tuple of single-byte symbols and stores it in `pre_token_tuple_counts`, weighted by its frequency. These byte-level sequences are the representation used for counting merge pairs in BPE training.

In [ ]:
vocab_size = 500
pre_token_tuple_counts = {}
merges = []

# Convert each pre-token (bytes) into a tuple of single-byte symbols,
# weighted by its frequency. This is the representation used for BPE pair counting.
for token_bytes, count in pre_token_counts.items():
    token_tuple = tuple(token_bytes[i : i+1] for i in range(len(token_bytes)))
    pre_token_tuple_counts[token_tuple] = pre_token_tuple_counts.get(token_tuple, 0) + count

print (pre_token_tuple_counts)
    

Then we iterate over each byte-level token in `pre_token_tuple_counts` and count how often each adjacent byte pair appears, weighted by token frequency. The result, `pair_freq`, records which byte pairs are most common and therefore candidates for BPE merges.

In [ ]:
## Count frequencies of adjacent byte pairs (within pre-tokens) for BPE merging
pair_freq = Counter()
for token_tuple, count in pre_token_tuple_counts.items():
    if len(token_tuple) < 2:
        continue
    for i in range(len(token_tuple) - 1):
        pair = (token_tuple[i], token_tuple[i+1])
        pair_freq[pair] += count

print (pair_freq)

Then we compute the maximum number of BPE merge operations allowed. The result, `max_merges`, limits how many merges can be learned.

In [ ]:
## Calculate maximum allowed merge operations:
# The final vocabulary consists of:
#   256 initial byte tokens + len(special_tokens) + number_of_merges
# So we can rearrange:
max_merges = vocab_size - len(special_tokens) - 256

print (max_merges)

##### BPE Training Loop

Finally, the training loop runs until `len(merges) == max_merges`: it selects the most frequent adjacent pair from `pair_freq` as `best_pair` and appends it to `merges`. There is a little more here, but this is the core of the idea. 

In [ ]:
while len(merges) < max_merges:
    print ("current vocab size: ", len(vocab))
    # Find the most frequent pair; in case of ties, choose the lexicographically greater pair.
    best_pair, best_count = None, 0
    for pair, freq in pair_freq.items():
        if freq > best_count or (freq == best_count and (best_pair is None or pair > best_pair)):
            best_pair, best_count = pair, freq
    
    # Record the merge 
    merges.append(best_pair)
    print ("best pair to be merged: ", best_pair)

    # Update the vocabulary with the new merged token 
    merged_token = best_pair[0] + best_pair[1]
    next_id = len(vocab)
    vocab[next_id] = merged_token

    # Update the tokens by merging occurrences of the best pair 
    new_token_tuple_counts = {}
    for token_tuple, count in pre_token_tuple_counts.items():
        i = 0
        while i < len(token_tuple) - 1:
            pair = (token_tuple[i], token_tuple[i+1])
            if pair == best_pair:
                prefix = token_tuple[:i]
                suffix = token_tuple[i+2 : ]
                token_tuple = prefix + (merged_token,) + suffix
            
                if prefix: 
                    left_pair = (prefix[-1], merged_token)
                    pair_freq[left_pair] = pair_freq.get(left_pair, 0) + count 
                    del_pair = (prefix[-1], best_pair[0])
                    pair_freq[del_pair] -= count 
                
                if suffix:
                    right_pair = (merged_token, suffix[0])
                    pair_freq[right_pair] = pair_freq.get(right_pair, 0) + count 
                    del_pair = (best_pair[1], suffix[0])
                    pair_freq[del_pair] -= count
            
                pair_freq[best_pair] -= count 
            i += 1
        new_token_tuple_counts[token_tuple] = count
    pre_token_tuple_counts = new_token_tuple_counts

    del pair_freq[best_pair]


#### Validation
We compare **our learned** BPE merges and vocabulary against a **reference** implementation by decoding GPT-2’s byte encoding and asserting that both the merges and vocab entries match.

In [ ]:
# Test our trained tokenizer with the references
reference_vocab_path = "data/train-bpe-reference-vocab.json"
reference_merges_path = "data/train-bpe-reference-merges.txt"

from util import gpt2_bytes_to_unicode

# Compare the learned merges to the expected output merges
gpt2_byte_decoder = {v: k for k, v in gpt2_bytes_to_unicode().items()}
with open(reference_merges_path) as f:
    gpt2_reference_merges = [tuple(line.rstrip().split(" ")) for line in f]
    reference_merges = [
        (
            bytes([gpt2_byte_decoder[token] for token in merge_token_1]),
            bytes([gpt2_byte_decoder[token] for token in merge_token_2]),
        )
        for merge_token_1, merge_token_2 in gpt2_reference_merges
    ]
assert merges == reference_merges


In [ ]:
import json 
# Compare the vocab to the expected output vocab
with open(reference_vocab_path) as f:
    gpt2_reference_vocab = json.load(f)
    reference_vocab = {
        gpt2_vocab_index: bytes([gpt2_byte_decoder[token] for token in gpt2_vocab_item])
        for gpt2_vocab_item, gpt2_vocab_index in gpt2_reference_vocab.items()
    }
# Rather than checking that the vocabs exactly match (since they could
# have been constructed differently, we'll make sure that the vocab keys and values match)
assert set(vocab.keys()) == set(reference_vocab.keys())
assert set(vocab.values()) == set(reference_vocab.values())

<a id="encoding_text"></a>
## Part 5. Encoding Text

The following two parts (Part 5: Encoding Text and Part 6: Decoding Text) are where you will implement the functions that will be graded:

* **Encoding** (10 points): Implementation of encoding functions in Part 5
  * `encode_chunk_no_special_tokens()`: 5 points (8 simple test cases like "hello")
  * `encode()` with special tokens: 5 points (1 test case with tinystories_sample.txt)
* **Decoding** (3 points): Implementation of `decode()` function in Part 6 (8 simple test cases like "hello")



In the previous part (BPE Tokenizer Training), we implemented a function to train a BPE tokenizer on input text
to obtain a tokenizer vocabulary and a list of BPE merges. Now, we will implement a BPE tokenizer that
loads a provided vocabulary and list of `merges` (BPE Tokenizer Training) from and uses them to encode and decode text to/from token IDs.

The process of encoding text with BPE mirrors training:

1. **Pre-tokenize:** Split text into pre-tokens and represent them as UTF-8 bytes.  
2. **Apply merges:** Apply the learned `merges` in the order they were created.  
3. **Special tokens:** Preserve user-defined special tokens during encoding.

Now let's use our tokenizer to encode some text from `tinystories_sample.txt`.

### Your Task: Implement BPE Encoding

You will implement **two encoding functions** that follow the steps above and convert text into token IDs using a trained BPE vocabulary and merge list.
You should reuse the `merges` Python variable learned during training and apply them in order (no retraining or frequency counting during encoding).

<a id="encode_chunk_no_special_tokens"></a>
#### **1. `encode_chunk_no_special_tokens`**

Encode a single text chunk **without special tokens**.

Your implementation should:

* Pre-tokenize the text using the same regex pattern (from `pattern`) as during training.
* Encode each pre-token to UTF-8 bytes (see [`Part 2. Unicode Encodings`](#unicode_encodings) for encoding) and split into individual bytes (creating a list of bytes called `parts`).
* Apply the BPE `merges` **in the same order as training** to `parts`.
* Map each merged byte sequence to token IDs using the vocabulary.

Assumptions:
* The input contains **no special tokens**.
* Only **one chunk** is encoded at a time.

You are given helper function `_apply_merge` already written below.


<a id="encode"></a>
#### **2. `encode`**

Encode a full text string **with special token support** (e.g., `<|endoftext|>`).

Your implementation should:

* If `special_tokens=False`, call `encode_chunk_no_special_tokens` directly.
* Otherwise:

  * Locate all occurrences of the special token. Consider a list of (start, end) tuples.
  * Split the text into chunks between special tokens.
  * Encode each chunk separately, using `encode_chunk_no_special_tokens.`
  * When encoding, insert the special token’s ID at the correct positions.

You may assume that `<|endoftext|>` is the only special token.


In [ ]:
from typing import Dict, List, Tuple, Set

# Helper function to apply merges
def _apply_merge(parts: List[bytes], pair: Tuple[bytes, bytes]) -> List[bytes]:
    """
    Apply a single merge operation to a sequence of byte parts.
    
    Args:
        parts (List[bytes]): The sequence of byte parts.
        pair (Tuple[bytes, bytes]): The pair of byte sequences to merge.
        
    Returns:
        List[bytes]: The sequence after applying the merge.
    """
    first, second = pair
    i = 0
    result = []

    while i < len(parts):
        # Try to find the first part of the pair 
        if i < len(parts) - 1 and parts[i] == first and parts[i+1] == second:
            # Merge the pair 
            result.append(first + second)
            i += 2
        else:
            # Keep the current part 
            result.append(parts[i])
            i += 1
    return result



In [ ]:
# encode a text chunk without special tokens
def encode_chunk_no_special_tokens(text, merges, vocab):
    """
    Encode a text string into a list of token IDs.
    Assume we are only getting one chunk at a time. 
    Assume there are no special tokens in the text.
    
    Args:
        text (str): The text to encode.
        merges (List[Tuple[bytes, bytes]]): The list of merges from BPE training.
        vocab (Dict[int, bytes]): The vocabulary from BPE training.
        
    Returns:
        List[int]: A list of token IDs.
    """
    # TODO: Pre-tokenize the text using the same regex pattern as during training
    # TODO: Process each pre-token:
    #   - Convert pre-token to bytes using .encode("utf-8")
    #   - Split into individual bytes
    #   - Apply the BPE merges from the learned `merges` list from training
    #   - Convert merged parts to token IDs using the vocab
    
    pass  # TODO: Remove this and implement the function


In [ ]:
def encode(text, merges, vocab, special_tokens=True):
    """
    Encode a text string into a list of token IDs, handling multiple special tokens.
    You may assume that `<|endoftext|>` is the ONLLY special token.
    
    Args:
        text (str): The text to encode.
        merges (List[Tuple[bytes, bytes]]): The list of merges from BPE training.
        vocab (Dict[int, bytes]): The vocabulary from BPE training.
        special_tokens: Whether to handle special tokens.
        
    Returns:
        List[int]: A list of token IDs.
    """
    if not special_tokens:
        return encode_chunk_no_special_tokens(text, merges, vocab)
    
    # TODO: Get the special token ID
    # TODO: Locate all positions of the special token in the text. Consider keeping track of a list of (start, end) tuples
    # TODO: Encode each chunk (text between located special tokens) separately using `encode_chunk_no_special_tokens`
    # TODO: When encoding, insert special token's ID at the appropriate positions
    
    pass  # TODO: Remove this and implement the function


Now let's put everything together to encode some text from `tinystories_sample.txt`.

In [ ]:
corpus_path = "data/tinystories_sample.txt"

with open(corpus_path, "r") as f:
    corpus_contents = f.read()

ids = encode(corpus_contents, merges, vocab, special_tokens=True)

print(ids)


<a id="decoding_text"></a>
## Part 6. Decoding Text

### Your Task: Implement Decoding

Now that we have our encoded `ids` we will decode them back into raw text.

<a id="decode"></a>
#### **`decode`**

To decode a sequence of **integer token IDs** back to **raw text**, we:
1. Look up each ID’s corresponding entries in the vocabulary (a byte sequence)
2. Concatenate them together
3. Decode the bytes to a Unicode string (see [`Part 2. Unicode Encodings`](#unicode_encodings) for decoding)

Note that input IDs are not guaranteed to map to valid Unicode strings (since a user
could input any sequence of integer IDs). The additional argument of `errors='replace'` in `.decode()` handles this case and replaces invalid byte sequences with the Unicode replacement character (U+FFFD).

Tip: Initialize a variable with `b''` (not a regular string `''`) when building byte sequences

In [ ]:
def decode(token_ids, vocab):
    """
    Decode a list of token IDs back to a text string.
    
    Args:
        token_ids (List[int]): The list of token IDs to decode.
        vocab (Dict[int, bytes]): The vocabulary from BPE training.
        
    Returns:
        str: The decoded text.
    """
    # TODO: Convert each token ID to its corresponding bytes using the vocab
    # TODO: Concatenate all the bytes together
    # TODO: Decode the bytes back to a UTF-8 string
    
    return "" # TODO: Remove this and implement the function


Let's try our function; we should get the story from `tinystories_sample.txt`!

In [ ]:
print (decode(ids, vocab))

<a id="ending_remarks"></a>
# Ending Remarks

Congratulations, you are done with the assignment!
Refer to the [`Submitting`](#submitting) for submission instructions.